# Part B Multi-Preprocess Count Model

Count regression with preprocessing comparison, TTA, and validation-based strategy selection.


In [1]:
class Cfg:
 z:tuple[int,int]=(416,416); bs:int=8; ep:int=30; lr:float=2.5e-4; wd:float=1e-4; vf:float=0.15; modes:tuple[str,...]=('fused',); tta:bool=True; seed:int=42
 bounds:tuple[float,float]=(90.0,220.0); max_scale:float=1.15
cfg=Cfg(); cfg


In [2]:
import math, random
import cv2, h5py, numpy as np, pandas as pd, torch
from copy import deepcopy
from pathlib import Path
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
R=globals().get('R',Path.cwd())
if not (R/'DATASET').exists() and (R.parent/'DATASET').exists(): R=R.parent
D=globals().get('D',R/'DATASET')
M=globals().get('M',R/'processed_density_maps')
if not M.exists() and (R/'ground_truth').exists(): M=R/'ground_truth'
OUT=globals().get('OUT',R/'prediction_results_part_b.csv')
DEV=globals().get('DEV',torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
def seed(x):
 random.seed(x); np.random.seed(x); torch.manual_seed(x)
 if torch.cuda.is_available(): torch.cuda.manual_seed_all(x)
def samples(split='train'):
 xs=[]; idir=D/'part_B'/('train_data' if split=='train' else 'test_data')/'images'; ddir=M/'part_B'/split
 for ip in sorted(idir.glob('IMG_*.jpg')):
  hp=ddir/f'{ip.stem}.h5'
  if hp.exists(): xs.append({'split':split,'image_id':ip.stem,'image_path':ip,'density_path':hp})
 return xs
img=lambda p: np.asarray(Image.open(p).convert('RGB'),dtype=np.uint8)
def den(p):
 with h5py.File(p,'r') as h: return h['density'][:].astype(np.float32)
cnt=lambda s: float(den(s['density_path']).sum())
def split(xs,vf=0.15,seedv=42):
 xs=list(xs); cs=[cnt(x) for x in xs]; xs=[x for _,x in sorted(zip(cs,xs),key=lambda t:t[0])]; step=max(2,round(1/max(vf,1e-6))); off=seedv%step
 return [x for i,x in enumerate(xs) if i%step!=off],[x for i,x in enumerate(xs) if i%step==off]
def bucket(c,bounds=None):
 b=cfg.bounds if bounds is None else bounds
 if c<b[0]: return 0
 if c<b[1]: return 1
 return 2
def prep(x,mode):
 rgb=x.astype(np.float32)/255.0
 g=cv2.cvtColor(x,cv2.COLOR_RGB2GRAY); c=np.repeat(cv2.createCLAHE(2.0,(8,8)).apply(g)[...,None],3,2).astype(np.float32)/255.0
 if mode=='rgb': return rgb
 if mode=='gray_clahe': return c
 return np.concatenate([rgb,c],2)
class DS(Dataset):
 def __init__(self,xs,mode,z=(416,416),aug=False): self.xs=list(xs); self.mode=mode; self.z=z; self.aug=aug
 def __len__(self): return len(self.xs)
 def __getitem__(self,i):
  s=self.xs[i]; x=img(s['image_path']); c=cnt(s); b=bucket(c)
  if self.aug and random.random()<0.5: x=np.ascontiguousarray(np.fliplr(x))
  if self.aug and random.random()<0.3: x=np.ascontiguousarray(np.flipud(x))
  if self.aug and random.random()<0.5: x=np.clip(x.astype(np.float32)*random.uniform(.9,1.1)+random.uniform(-12,12),0,255).astype(np.uint8)
  h,w=x.shape[:2]
  if self.aug and min(h,w)>280 and random.random()<0.45:
   ch=random.randint(int(.78*h),h); cw=random.randint(int(.78*w),w); t=random.randint(0,h-ch); l=random.randint(0,w-cw); x=x[t:t+ch,l:l+cw]
  x=cv2.resize(x,self.z,interpolation=cv2.INTER_AREA)
  return torch.from_numpy(prep(x,self.mode).transpose(2,0,1)), torch.tensor([math.log1p(c)],dtype=torch.float32), torch.tensor([c],dtype=torch.float32), torch.tensor(b,dtype=torch.long)
class SE(nn.Module):
 def __init__(self,c,r=8):
  super().__init__(); h=max(8,c//r); self.m=nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Conv2d(c,h,1),nn.ReLU(True),nn.Conv2d(h,c,1),nn.Sigmoid())
 def forward(self,x): return x*self.m(x)
class RB(nn.Module):
 def __init__(self,a,b,s=1):
  super().__init__(); self.p=nn.Identity() if (a==b and s==1) else nn.Sequential(nn.Conv2d(a,b,1,s,0),nn.BatchNorm2d(b))
  self.m=nn.Sequential(nn.Conv2d(a,b,3,s,1),nn.BatchNorm2d(b),nn.ReLU(True),nn.Conv2d(b,b,3,1,1),nn.BatchNorm2d(b),SE(b)); self.r=nn.ReLU(True)
 def forward(self,x): return self.r(self.p(x)+self.m(x))
class Net(nn.Module):
 def __init__(self):
  super().__init__(); ch=6 if 'fused' in cfg.modes else 3
  self.m=nn.Sequential(RB(ch,32,2),RB(32,64,2),RB(64,96,1),RB(96,128,2),RB(128,160,1),RB(160,224,2))
  self.p=nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Flatten())
  self.h=nn.Sequential(nn.Linear(224,128),nn.ReLU(True),nn.Dropout(.2),nn.Linear(128,1))
  self.r=nn.Sequential(nn.Linear(224,96),nn.ReLU(True),nn.Dropout(.1),nn.Linear(96,1))
  self.c=nn.Sequential(nn.Linear(224,64),nn.ReLU(True),nn.Dropout(.1),nn.Linear(64,3))
 def forward(self,x,aux=False):
  f=self.m(x); g=self.p(f); y=self.h(g); r=self.r(g); z=y+.15*r
  if aux: return z,self.c(g)
  return z
def ep(model,ld,opt=None):
 tr=opt is not None; model.train(tr); reg=nn.SmoothL1Loss(beta=.22,reduction='none'); mse=nn.MSELoss(reduction='none'); clf=nn.CrossEntropyLoss(); tl=ta=trm=n=0
 for x,y,c,b in ld:
  x=x.to(DEV); y=y.to(DEV); c=c.to(DEV).reshape(-1); b=b.to(DEV)
  with torch.set_grad_enabled(tr):
   z,logits=model(x,True); base=reg(z.reshape(-1),y.reshape(-1)); cntp=torch.expm1(z.reshape(-1)); w=(1.0+0.0025*c).clamp_max(2.8); loss=(base*w).mean()+0.12*((mse(cntp,c)/(20.0+c))*w).mean()+0.10*clf(logits,b)
   if tr: opt.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); opt.step()
  p=cntp.detach(); e=p-c; bs=x.size(0); tl+=float(loss.detach())*bs; ta+=float(e.abs().sum()); trm+=float((e**2).sum()); n+=bs
 return {'loss':tl/max(n,1),'mae':ta/max(n,1),'rmse':math.sqrt(trm/max(n,1))}
def infer_once(model,x,mode):
 q=cv2.resize(x,cfg.z,interpolation=cv2.INTER_AREA); u=torch.from_numpy(prep(q,mode).transpose(2,0,1)).unsqueeze(0).to(DEV)
 with torch.no_grad(): z,logits=model(u,True)
 return float(torch.expm1(z.reshape(-1)).cpu().item()), torch.softmax(logits,1).cpu().numpy()[0].astype(np.float32)
def pred_stats(model,pth,mode):
 x=img(pth); xs=[x]
 if cfg.tta:
  for s in (0.9,1.0,cfg.max_scale):
   h=max(32,int(round(x.shape[0]*s))); w=max(32,int(round(x.shape[1]*s))); q=cv2.resize(x,(w,h),interpolation=cv2.INTER_AREA if s<=1 else cv2.INTER_CUBIC); xs.append(q); xs.append(np.ascontiguousarray(np.fliplr(q)))
 ps=[]; bs=[]
 for q in xs:
  c,p=infer_once(model,q,mode); ps.append(c); bs.append(p)
 raw=float(np.mean(ps)); prob=np.mean(np.stack(bs,0),0); return {'raw':raw,'bucket':int(np.argmax(prob))}
def fit_affine(a,p):
 x=np.asarray(p,np.float64); y=np.asarray(a,np.float64); m=np.isfinite(x)&np.isfinite(y)
 x=x[m]; y=y[m]
 if len(x)<3: return {'a':1.0,'b':0.0}
 q1,q9=np.quantile(x,[.05,.95]); m=(x>=q1)&(x<=q9); x=x[m]; y=y[m]
 A=np.vstack([x,np.ones_like(x)]).T; sol=np.linalg.lstsq(A,y,rcond=None)[0]; aa=float(np.clip(sol[0],0.7,1.35)); bb=float(np.clip(sol[1],-25.0,35.0))
 return {'a':aa,'b':bb}
def score(a,p):
 e=np.abs(p-a); med=float(np.median(e)); trim=float(np.sort(e)[:max(1,int(math.ceil(.9*len(e))))].mean()); p90=float(np.quantile(e,.9)); bias=float((p-a).mean()); mae=float(e.mean()); big=float(np.maximum(e-80,0).mean())
 s=.4*med+.25*trim+.15*p90+.1*abs(bias)+.1*big
 return (s,med,p90,abs(bias)),{'val_score':s,'val_mae':mae,'val_medae':med,'val_p90ae':p90,'val_trimmed_mae':trim,'val_bias':bias,'val_big_error':big}
def merge(fs):
 z=fs[0]
 for f in fs[1:]: z=z.merge(f,on=['image_id','actual_count'],how='inner')
 return z
def choose(v,modes):
 mode=modes[0]; a=v.actual_count.to_numpy(np.float32); raw=v[f'raw_{mode}'].to_numpy(np.float32); aff=fit_affine(a,raw); rows=[]; best=None
 opts=[{'name':f'single_{mode}','cal':{'kind':'identity'}},{'name':f'single_{mode}','cal':{'kind':'affine','a':aff['a'],'b':aff['b']}}]
 for o in opts:
  if o['cal']['kind']=='identity': adj=np.maximum(raw,0.0)
  else: adj=np.maximum(raw*o['cal']['a']+o['cal']['b'],0.0)
  sc,mt=score(a,adj); r={'strategy':o['name'],'blend_alpha':None,'calibrator':o['cal']['kind']}; r.update(mt); rows.append(r)
  if best is None or sc<best[0]: best=(sc,{'strategy':o['name'],'blend_alpha':None,'calibrator':o['cal']})
 rows=sorted(rows,key=lambda r:(r['val_score'],r['val_p90ae'],r['val_medae'],abs(r['val_bias']))); return best[1],pd.DataFrame(rows).reset_index(drop=True)
def apply_row(r,st,modes):
 raw=max(0.0,float(r[f'raw_{modes[0]}'])); c=st['calibrator']
 if c.get('kind')=='affine': return max(0.0,float(raw*c['a']+c['b']))
 return raw


In [3]:
seed(cfg.seed)
tr_all=samples('train'); te=samples('test'); tr,va=split(tr_all,cfg.vf,cfg.seed)
mods={}; h=[]; vf=[]
for i,mode in enumerate(cfg.modes):
 tl=DataLoader(DS(tr,mode,cfg.z,True),batch_size=cfg.bs,shuffle=True,num_workers=0); vl=DataLoader(DS(va,mode,cfg.z,False),batch_size=cfg.bs,shuffle=False,num_workers=0)
 m=Net().to(DEV); opt=torch.optim.AdamW(m.parameters(),lr=cfg.lr,weight_decay=cfg.wd); sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode='min',factor=.5,patience=3); best=deepcopy(m.state_dict()); bestv=float('inf'); rows=[]
 for e in range(1,cfg.ep+1):
  tm=ep(m,tl,opt); vm=ep(m,vl,None); sch.step(vm['mae']); d={'mode':mode,'epoch':e,'lr':opt.param_groups[0]['lr']}; d.update({f'train_{k}':v for k,v in tm.items()}); d.update({f'val_{k}':v for k,v in vm.items()}); rows.append(d)
  if vm['mae']<bestv: bestv=vm['mae']; best=deepcopy(m.state_dict())
 m.load_state_dict(best); mods[mode]=m; h.append(pd.DataFrame(rows)); vals=[]
 for s in va:
  z=pred_stats(m,s['image_path'],mode); vals.append({'image_id':s['image_id'],'actual_count':cnt(s),f'raw_{mode}':z['raw'],f'bucket_{mode}':z['bucket']})
 vf.append(pd.DataFrame(vals))
hist=pd.concat(h,ignore_index=True); v=merge(vf); st,stab=choose(v,cfg.modes); v['raw_predicted_count']=v.apply(lambda r: apply_row(r,{'strategy':st['strategy'],'blend_alpha':st['blend_alpha'],'calibrator':{'kind':'identity'}},cfg.modes),axis=1); v['predicted_count']=v.apply(lambda r: apply_row(r,st,cfg.modes),axis=1); v['absolute_error']=(v.predicted_count-v.actual_count).abs(); print('train',len(tr),'val',len(va),'test',len(te)); print('strategy',st); display(hist.groupby('mode').tail(3)); display(stab.head(10)); display(v.head())
tf=[]
for mode in cfg.modes:
 rows=[]
 for s in te:
  z=pred_stats(mods[mode],s['image_path'],mode); rows.append({'image_id':s['image_id'],'actual_count':cnt(s),f'raw_{mode}':z['raw'],f'bucket_{mode}':z['bucket']})
 tf.append(pd.DataFrame(rows))
p=merge(tf); p['strategy_name']=st['strategy']; p['blend_alpha']=st['blend_alpha']; p['predicted_count']=p.apply(lambda r: apply_row(r,st,cfg.modes),axis=1)
p['raw_predicted_count']=p.apply(lambda r: apply_row(r,{'strategy':st['strategy'],'blend_alpha':st['blend_alpha'],'calibrator':{'kind':'identity'}},cfg.modes),axis=1)
p['signed_error']=p.predicted_count-p.actual_count; p['absolute_error']=p.signed_error.abs(); p.insert(0,'part','B'); p.insert(1,'split','test'); p['image_path']=p.image_id.map(lambda x:f'DATASET/part_B/test_data/images/{x}.jpg'); p=p[['part','split','image_id','image_path','actual_count','strategy_name','blend_alpha','raw_predicted_count','predicted_count','signed_error','absolute_error']]; p.to_csv(OUT,index=False); print('saved',OUT); display(p.head())


train 342 val 58 test 316
strategy {'strategy': 'single_fused', 'blend_alpha': None, 'calibrator': {'kind': 'identity'}}


,mode,epoch,lr,train_loss,train_mae,train_rmse,val_loss,val_mae,val_rmse
27,fused,28,0.000016,5.283841,49.522908,76.075037,2.879020,35.184759,60.001280
28,fused,29,0.000016,7.054885,54.037631,87.855938,3.048036,34.958435,63.030689
29,fused,30,0.000016,5.082818,49.686039,73.328989,2.852847,36.117651,55.990650


,strategy,blend_alpha,calibrator,val_score,val_mae,val_medae,val_p90ae,val_trimmed_mae,val_bias,val_big_error
0,single_fused,None,identity,26.658719,34.741390,14.709915,84.309337,21.315031,-19.453564,8.542380
1,single_fused,None,affine,27.211245,34.329132,19.670658,83.609520,23.938747,-2.429525,5.739145


,image_id,actual_count,raw_fused,bucket_fused,raw_predicted_count,predicted_count,absolute_error
0,IMG_158,12.000001,24.656342,0,24.656342,24.656342,12.656341
1,IMG_93,19.999996,30.336178,0,30.336178,30.336178,10.336181
2,IMG_373,22.999998,27.878605,0,27.878605,27.878605,4.878607
3,IMG_276,26.000002,34.136335,0,34.136335,34.136335,8.136333
4,IMG_350,27.999994,30.695102,0,30.695102,30.695102,2.695107


saved c:\Users\pendy\Desktop\s6\projects\dip\prediction_results_part_b.csv


,part,split,image_id,image_path,actual_count,strategy_name,blend_alpha,raw_predicted_count,predicted_count,signed_error,absolute_error
0,B,test,IMG_1,DATASET/part_B/test_data/images/IMG_1.jpg,23.000002,single_fused,None,27.912261,27.912261,4.912259,4.912259
1,B,test,IMG_10,DATASET/part_B/test_data/images/IMG_10.jpg,180.999985,single_fused,None,187.018186,187.018186,6.018202,6.018202
2,B,test,IMG_100,DATASET/part_B/test_data/images/IMG_100.jpg,156.999985,single_fused,None,101.296955,101.296955,-55.703030,55.703030
3,B,test,IMG_101,DATASET/part_B/test_data/images/IMG_101.jpg,36.999992,single_fused,None,42.785960,42.785960,5.785967,5.785967
4,B,test,IMG_102,DATASET/part_B/test_data/images/IMG_102.jpg,69.999992,single_fused,None,85.902685,85.902685,15.902693,15.902693
